In [ ]:
# STEP 1: Mount Google Drive

# Import the Google Drive module from Google Colab
# This allows the notebook to access files stored in your Google Drive
from google.colab import drive

# Connect (mount) your Google Drive to the Colab environment
# After this, files in Drive can be accessed like a normal folder
drive.mount('/content/drive')


# STEP 2: Import libraries

# Import OS library
# Used to work with folder paths and directories
import os

# Import matplotlib for creating graphs
# This will help visualize training results like accuracy and loss
import matplotlib.pyplot as plt

# Import pandas for handling structured data and plotting training metrics
import pandas as pd

# Import ImageDataGenerator
# This tool loads images from folders and prepares them for model training
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Import EfficientNetB0 model
# EfficientNetB0 is a powerful pre-trained image classification model
# It already learned general image patterns from millions of images
from tensorflow.keras.applications import EfficientNetB0

# Import Model class
# Used to build a custom neural network architecture
from tensorflow.keras.models import Model

# Import neural network layers used to build the classification head
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout

# Import Adam optimizer
# This controls how the model updates its knowledge during training
from tensorflow.keras.optimizers import Adam


# STEP 3: Define paths

# Define the main dataset folder location inside Google Drive
# This folder contains both training and validation datasets
base_dir = '/content/drive/MyDrive/ML_WSSV/WSSV/dataset/Augmented_data'

# Define the path for training images
# These images are used to teach the model how to recognize shrimp health conditions
train_dir = os.path.join(base_dir, 'train')

# Define the path for validation images
# These images are used to test how well the model learned
val_dir = os.path.join(base_dir, 'validation_data')

# Define where the trained model will be saved in Google Drive
# This file will later be used for shrimp disease prediction
model_save_path = '/content/drive/MyDrive/ML_WSSV/WSSV/Trained_Model/efficientnetb0_model.h5'


# STEP 4: Data Generators

# Create a training data generator
# rescale=1./255 converts pixel values from 0-255 to 0-1
# This normalization helps the model learn more effectively
train_datagen = ImageDataGenerator(rescale=1./255)

# Create a validation data generator
# It performs the same normalization for validation images
val_datagen = ImageDataGenerator(rescale=1./255)


# Load training images from the directory
# target_size=(224,224) resizes images to the required input size for EfficientNet
# batch_size=32 means 32 images are processed at a time
# class_mode='binary' because we have two classes (Healthy and WSSV)
train_gen = train_datagen.flow_from_directory(train_dir, target_size=(224, 224), batch_size=32, class_mode='binary')

# Load validation images from the directory
# These images will be used to evaluate model performance during training
val_gen = val_datagen.flow_from_directory(val_dir, target_size=(224, 224), batch_size=32, class_mode='binary')


# STEP 5: Build EfficientNetB0 model

# Load EfficientNetB0 model with pre-trained weights from ImageNet dataset
# include_top=False removes the original classification layers
# input_shape defines the expected image size and color channels
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze the base model layers
# This keeps the learned image features unchanged during training
# Only the new layers we add will learn shrimp-specific patterns
base_model.trainable = False


# Get the output features from the EfficientNet base model
x = base_model.output

# Apply Global Average Pooling
# This converts feature maps into a single vector for classification
x = GlobalAveragePooling2D()(x)

# Add a fully connected layer with 128 neurons
# This layer helps the model learn shrimp disease features
x = Dense(128, activation='relu')(x)

# Add dropout layer
# This randomly disables some neurons during training
# It helps prevent the model from memorizing the training data (overfitting)
x = Dropout(0.3)(x)

# Final output layer
# Since this is binary classification (Healthy vs WSSV)
# we use one neuron with sigmoid activation
predictions = Dense(1, activation='sigmoid')(x)


# Create the final model combining base model and new layers
model = Model(inputs=base_model.input, outputs=predictions)


# Compile the model
# optimizer controls learning adjustments
# loss measures prediction error
# accuracy tracks how many predictions are correct
model.compile(optimizer=Adam(learning_rate=0.0001), loss='binary_crossentropy', metrics=['accuracy'])


# STEP 6: Train the model

# Start training the model using the training dataset
# epochs=10 means the entire dataset will be used 10 times for learning
# validation_data checks performance using unseen validation images
history = model.fit(train_gen, epochs=10, validation_data=val_gen)


# STEP 7: Save the model

# Save the trained model to Google Drive
# This allows the model to be reused later for predictions
model.save(model_save_path)

# Print confirmation message showing where the model was saved
print(f"✅ Model saved to: {model_save_path}")


# STEP 8: Plot accuracy and loss

# Plot training accuracy vs validation accuracy
# This graph shows how well the model learned over time
pd.DataFrame(history.history)[['accuracy', 'val_accuracy']].plot()

# Add title to the accuracy graph
plt.title('Model Accuracy')

# Display the accuracy graph
plt.show()


# Plot training loss vs validation loss
# Loss indicates how far predictions are from the correct answer
pd.DataFrame(history.history)[['loss', 'val_loss']].plot()

# Add title to the loss graph
plt.title('Model Loss')

# Display the loss graph
plt.show()

Mounted at /content/drive
Found 1133 images belonging to 2 classes.
Found 282 images belonging to 2 classes.
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 492s 13s/step - accuracy: 0.6475 - loss: 0.6524 - val_accuracy: 0.6525 - val_loss: 0.6464
Epoch 2/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 144s 3s/step - accuracy: 0.6506 - loss: 0.6529 - val_accuracy: 0.6525 - val_loss: 0.6459
Epoch 3/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 135s 4s/step - accuracy: 0.6484 - loss: 0.6513 - val_accuracy: 0.6525 - val_loss: 0.6464
Epoch 4/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 134s 4s/step - accuracy: 0.6471 - loss: 0.6496 - val_accuracy: 0.6525 - val_loss: 0.6466
Epoch 5/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 131s 3s/step - accuracy: 0.6505 - loss: 0.6515 - val_accuracy: 0.6525 - val_loss: 0.6485
Epoch 6/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 117s 3s/step - accuracy: 0.6540 - loss: 0.6474 - val_accuracy: 0.6525 - val_loss: 0.6459
Epoch 7/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 115s 3s/step - accuracy: 0.6647 - loss: 0.6425 - val_accuracy: 0.6525 - val_loss: 0.6462
Epoch 8/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 117s 3s/step - accuracy: 0.6541 - loss: 0.6462 - val_accuracy: 0.6525 - 